#Instalação de libs

In [0]:
!pip install -q openpyxl xlsxwriter

In [0]:
dbutils.library.restartPython()

#Imports

In [0]:
import os
import pandas as pd
import openpyxl
import copy
import json
import requests
import subprocess

from dataclasses import dataclass, asdict
from datetime import date, datetime
from pathlib import Path

In [0]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 230)

#Parâmetros

In [0]:
dbutils.widgets.text("id_projeto", "colon", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
print("environment:", environment)

In [0]:
environment_tbl = "" if environment in ["hml", "prd"] else f"{environment}_"
print("environment_tbl:", environment_tbl)

In [0]:
dbutils.widgets.text("catalog", "ia", "Catalog")
catalog_name = dbutils.widgets.get("catalog")
print(f"catalog_name: {catalog_name}")

In [0]:
dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.now().strftime("%Y-%m-%d")
print(f"Data Referencia: {data_execucao_modelo}")

In [0]:
if environment in ["hml", "prd"]:
    main_catalog = catalog_name + "."
else:
    main_catalog = catalog_name + "." + f"{environment_tbl}"

print(f"main_catalog: {main_catalog}")

In [0]:
# if schema_name == "":
#     root_folder = f"/mnt/trusted/datalake/{main_catalog}/data/{id_projeto}/diamond"
# else:
#     root_folder = f"abfss://artificial-intelligence@sardslusdevelopmenthml.dfs.core.windows.net/curated/ia/diamond/{id_projeto}/{environment}"

# print(root_folder)

In [0]:
root_remote_path = f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/"
print(root_remote_path)

In [0]:
root_remote_config_path = f"{root_remote_path}config/{environment}/"
print(root_remote_config_path)

In [0]:
root_remote_data_path = f"{root_remote_path}data/{environment}/envio/"
print(root_remote_data_path)

In [0]:
year, month, day = data_execucao_modelo.split("-")
remote_path = f"{root_remote_data_path}{year}/{month}/"
print(remote_path)

In [0]:
current_folder = os.path.join("/tmp", id_projeto) + "/"

Path(current_folder).mkdir(parents=True, exist_ok=True)

print(current_folder)

In [0]:
ls -lha {current_folder}

# Funções Auxiliares

In [0]:
# def optimize_table(table_id):
#     spark.sql(f"VACUUM {table_id}")
#     spark.sql(f"OPTIMIZE {table_id}")
#     spark.sql(f"ANALYZE TABLE {table_id} COMPUTE STATISTICS")

In [0]:
# table_location = lambda x: f"{root_folder}/{x.split('.')[-1]}"

#Variáveis com os nomes das tabelas

In [0]:
tbl_saida = f"{main_catalog}tb_diamond_mod_colon_saida"

In [0]:
print(f"{'tbl_saida':15}: {tbl_saida}")

# Exporta os dados para Excel

In [0]:
def get_data(unidades):
  df = spark.sql(f"""
      with cte_input as (
            select
                  dataExecucaoModelo,
                  fl_relevante as flgRelevante,
                  exm_laudo_achados as laudoExameAchado,
                  exm_laudo_resultado as laudoExameResultado,
                  exm_laudo_texto_tratado as laudoExameTratado,
                  exm_laudo_texto as laudoExameOriginal,
                  exm_laudo_dataliber as laudoExameDataLiberacao,
                  exm_status as exameStatus,
                  exm_an as exameAn,
                  exm_tipo as tipoExame,
                  exm_titulo as exameTitulo,
                  exm_mod as exameMod,
                  exm_data as exameData,
                  gold_corporativo.default.rdsl_decrypt(telefonePaciente,0) as telefonePaciente,
                  telefonePacienteDDD as telefonePacienteDDD,
                  idade_paciente as idadePaciente,
                  pct_datanasc as dataNascimentoPaciente,
                  pct_sexo as sexoPaciente,
                  gold_corporativo.default.rdsl_decrypt(pct_cpf,1) as cpfPaciente,
                  gold_corporativo.default.rdsl_decrypt(pct_nome,1) as nomePaciente,
                  unidade as nomeHospital,
                  idunidade as idHospital,
                  id_pct as idPaciente,
                  record_id as recordId,
                  nme_regional_hospital as ufRegional
            from {tbl_saida}
            where dataExecucaoModelo = date('{data_execucao_modelo}')
             and fl_relevante = 1
             and idunidade in ({unidades})
      )
      ,cte as (
            select
                dataExecucaoModelo,
                ufRegional,
                nomeHospital,
                nomePaciente,
                cpfPaciente, 
                sexoPaciente,
                idadePaciente,
                telefonePaciente, 
                exameAn,
                laudoExameDataLiberacao,
                tipoExame,
                laudoExameOriginal,
                laudoExameResultado,
                laudoExameAchado,
                exameMod,
                exameData,
                exameTitulo,

                  '' as achado_relevante,
                  '' as oncologia, 
                  '' as comentarios,
                  '' as cadatro_convenio,
                  '' as cadastro_plano,
                  '' as data_liberacao_exame,
                  '' as material,
                  '' as data_1_contato_MSG,
                  '' as data_2_contato_tel,
                  '' as data_3_contato_MSG,
                  '' as paciente_navegado,
                  '' as data_admissao_navegacao,
                  '' as mes,
                  '' as ano,
                  '' as data_alta_navegacao,
                  '' as motivo_alta_navegacao,
                  '' as diagnostico,
                  '' as status_1_consulta_especialista,
                  '' as data_1_consulta_especialista,
                  '' as nome_medico,
                  '' as unidade_gendamento,
                  '' as tipo_tratamento,
                  '' as cirugia_solicitada,
                  '' as data_pedido,
                  '' as data_cirugia,
                  '' as local_cirurgia,
                  '' as observacoes,

                  "" as achadoRelevante,
                  -- "" as linhaCuidado,
                  "" as destino,
                  "" as prioridade,

                  case 
                        when trim(upper(ufRegional)) = 'RJ' then 1
                        when trim(upper(ufRegional)) = 'SP' then 2
                        when trim(upper(ufRegional)) = 'BA' then 3
                        else 4
                  end as regionalHospitalOrder
            from cte_input
      )
      select * except(regionalHospitalOrder)
      from cte
    --   where idadePaciente <= 80 removido apos conversar com Natan 2026-01-22
      order by
        regionalHospitalOrder,
        nomePaciente,
        exameData

  """).toPandas()

  return df

In [0]:
# ['nme_regional_hospital', 'exameData', 'laudoExameOriginal', 'laudoExameAchado', 'exameMod', 'exameTitulo'] 

In [0]:
%sql
select *
from ia.dev_tb_diamond_mod_colon_saida
limit 100

In [0]:
# def get_data(unidades):
#   df = spark.sql(f"""
#     with cte_input as (
#         select
#             dataExecucaoModelo,
#             record_id as idPredicao,
#             exm_an as idExame,
#             pct_nome as nomePaciente,
#             pct_datanasc as dataNascimentoPaciente,
#             idade_paciente as idadePaciente,
#             nme_convenio as nomeConvenio,
#             idunidade as idHospital, 
#             unidade as nomeHospital,
#             nme_regional_hospital as regionalHospital,
#             exm_laudo_dataliber as dataExame,
#             exm_titulo as tipoExame,
#             exm_laudo_texto_tratado as laudoExameOriginal,
#             exm_laudo_resultado as laudoResultado,
#             exm_laudo_achados as laudoAchado,
#             fl_relevante as flgRelevante
#         from {tbl_saida}
#         where dataExecucaoModelo = date('{data_execucao_modelo}')
#           and fl_relevante = 1
#           and idunidade in ({unidades})
#           and nme_convenio not rlike r'(?i)(?<!\\w)AMIL(?:\\s*[-/().A-Za-z0-9]+)?'
#     ),
#     cte as (
#         select
#             dataExecucaoModelo,
#             idPredicao,
#             idExame,
#             nomePaciente,
#             idadePaciente,
#             nomeHospital,
#             regionalHospital,
#             date_format(date(dataExame), 'dd/MM/yyyy') as dataExame,
#             trim(REGEXP_REPLACE(tipoExame, '[^A-Za-z0-9\\s\\.,;!?\\-]+', '')) as tipoExame,
#             laudoExameOriginal as laudoExame,
#             laudoAchado,
#             "" as achadoRelevante,
#             "" as achado,
#             "" as linhaCuidado,
#             "" as observacao,
#             "" as destino,
#             "" as prioridade
#         from cte_input
#     )
#     select *
#     from cte
#     order by regionalHospital, nomePaciente, dataExame
#   """).toPandas()

#   return df


In [0]:
# def get_data(unidades):
#   df = spark.sql(f"""
#         select
#             dataExecucaoModelo,
#             fl_relevante as flgRelevante,
#             exm_laudo_achados as laudoExameAchado,
#             exm_laudo_resultado as laudoExameResultado,
#             exm_laudo_texto_tratado as laudoExameTratado,
#             exm_laudo_texto as laudoExameOriginal,
#             exm_laudo_dataliber as laudoExameDataLiberacao,
#             exm_status as exameStatus,
#             exm_an as exameAn,
#             exm_tipo as tipoExame,
#             exm_titulo as exameTitulo,
#             exm_mod as exameMod,
#             exm_data as exameData,
#             telefonePaciente as telefonePaciente,
#             telefonePacienteDDD as telefonePacienteDDD,
#             idade_paciente as idadePaciente,
#             pct_datanasc as dataNascimentoPaciente,
#             pct_sexo as sexoPaciente,
#             pct_cpf as cpfPaciente,
#             gold_corporativo.default.rdsl_decrypt(pct_nome,1) as nomePaciente,
#             unidade as nomeHospital,
#             idunidade as idHospital,
#             id_pct as idPaciente,
#             record_id as recordId

#         from {tbl_saida}
#         where dataExecucaoModelo = date('{data_execucao_modelo}')
#           and fl_relevante = 1
#   """).toPandas()

#   return df

In [0]:
# def get_cols():
#     """Dicionário de/para com os nomes das colunas de saída"""
#     dic_col_names = {
#         "dataExecucaoModelo": "Data Referência",
#         "record_id": "idPredicao",
#         "idExame": "idExame",
#         "nomePaciente": "Nome Paciente",
#         "idadePaciente": "Idade Paciente",
#         "nomeHospital": "Nome Hospital",
#         "regionalHospital": "UF Hospital",
#         "nomeMedico": "Médico Solicitante",
#         "numCrm": "CRM",
#         "ufCrm": "UF CRM",
#         "dataExame": "Data Exame",
#         "tipoExame": "Tipo Exame",
#         "laudoExame": "Laudo",
#         "valorPlt": "Valor PLT",
#         "difTempoImagemPlt": "Dif. Tempo Imagem PLT",
#         "achadoRelevante": "Achado Relevante",
#         "achado": "Achado",
#         "linhaCuidado": "Linha de Cuidado",
#         "destino": "Destino",
#         "prioridade": "Prioridade",
#         "observacao": "Observação",
#     }

#     return dic_col_names

In [0]:
# Achado Relevante	Achado	Linha de Cuidado	Destino	Prioridade	Observação

#        "achadoRelevante": "Achado Relevante",
#         "achado": "Achado",
#         "linhaCuidado": "Linha de Cuidado",
#         "destino": "Destino",
#         "prioridade": "Prioridade",
#         "observacao": "Observação",

In [0]:
"['laudoExameOriginal', 'laudoExameAchado'] not in index"

In [0]:
# def get_cols():
#     """Dicionário de/para com os nomes das colunas de saída"""
#     dic_col_names = {
#         "dataExecucaoModelo": "Data Referência",
#         "regionalHospital": "Regional",
#         "nomePaciente": "Nome Paciente",
#         "idadePaciente": "Idade Paciente",
#         "tipoExame": "Origem Exame", 
#         "nomeHospital": "Nome Hospital", 
#         "dataExame": "Data Exame",  
#         "laudoExame": "Laudo", 
#         "laudoAchado": "Achado Laudo", 
#         "tipoExame": "Exame",
#         # "exameTitulo": "Tipo Exame",
#         ## 
#         "achadoRelevante": "Achado Relevante",
#         "achado": "Achado",
#         "linhaCuidado": "Linha de Cuidado",
#         "destino": "Destino",
#         "prioridade": "Prioridade",
#         "observacao": "Observação",
#     }

#     return dic_col_names

In [0]:
def get_cols():
    """Dicionário de/para com os nomes das colunas de saída"""
    dic_col_names = {
        "ufRegional" : "Regional",
        "nomeHospital": "Unidade", 
        "nomePaciente": "Nome Paciente",
        "cpfPaciente" : "CPF",
        "sexoPaciente" : "Sexo",
        "idadePaciente": "Idade",
        "telefonePaciente": "Telefone",
        "exameAn": "Codigo AN",
        "laudoExameDataLiberacao": "Data de Liberacao do Laudo",
        "tipoExame": "Origem Exame",
        "exameMod" : "Modalidade",
        "exameTitulo": "Tipo Exame",
        "laudoExameOriginal": "Laudo", 
        "laudoExameResultado": "Resultado", 
         "laudoExameAchado" : "Achados",
        "dataExecucaoModelo": "Data de Execucao do Modelo",

        "achado_relevante" : "Achado Relevante",
        "oncologia" : "Oncologia", 
        "comentarios" : "Comentarios",
        "cadatro_convenio" : "Cadastro Convenio",
        "cadastro_plano" : "Cadastro Plano",
        "data_liberacao_exame" : "Data de Liberacao de Exame",
        "material" : "Material",
        "data_1_contato_MSG" : "Data 1º Contato | MSG",
        "data_2_contato_tel" : "Data 2º Contato | Tel",
        "data_3_contato_MSG" : "Data 3º Contato | MSG",
        "paciente_navegado" : "Paciente Navegado",
        "data_admissao_navegacao" : "Data de Admissao na Navegacao",
        "mes" : "Mes",
        "ano" : "Ano",
        "data_alta_navegacao" : "Data da Alta da Navegacao",
        "motivo_alta_navegacao" : "Motivo da Alta da Navegacao",
        "diagnostico" : "Diagnostico",
        "status_1_consulta_especialista" : "Status da 1º Consulta com o Especialista",
        "data_1_consulta_especialista" : "Data 1º Consulta com o Especialista",
        "nome_medico" : "Nome do Medico",
        "unidade_gendamento" : "Unidade de Agendamento",
        "tipo_tratamento" : "Tipo de Tratamento",
        "cirugia_solicitada" : "Cirurgia Solicitada",
        "data_pedido" : "Data do Pedido",
        "data_cirugia" : "Data da Cirugia",
        "local_cirurgia" : "Local da Cirurgia",
        "observacoes" : "Observacoes",

        # "linhaCuidado": "Linha de Cuidado", removido apos conversa com Natan em 2026-01-23
    }

    return dic_col_names

In [0]:
def export_to_excel(df, dic_col_names, unidade):
    file_name = f"{id_projeto}_{unidade}_{data_execucao_modelo}.xlsx"
    local_file = f"{current_folder}{file_name}"
    remote_file = f"{remote_path}{file_name}"

    cols = [col for col in dic_col_names.keys()]
    df[cols].to_excel(local_file, index=False, engine='xlsxwriter')

    return {
        "unidade": unidade,
        "nomeArquivo": file_name,
        "arquivoLocal": local_file,
        "arquivoRemoto": remote_file,
        "caminhoRemoto": remote_path.replace("/mnt", ""),
        "registros": len(df.index),
        "dataProcessamento": data_execucao_modelo,
        "dataProcessamentoFormatada": datetime.strptime(data_execucao_modelo, "%Y-%m-%d").strftime("%d/%m/%Y"),
    }

In [0]:
def get_cols_config():
    """Define tamanho fixo para algumas colunas. ( -1 = Coluna oculta )"""
    dic_col_fixed_width = {
        "Data de Execucao do Modelo": -1,
        "Laudo": 150,
        "Achados": 80,
        "Resultado": 130,
        "Observações": 30,
        "Achado Relevante": 37,
        "Linha de Cuidado": 27,
        "Destino": 80,
        
    }
    return dic_col_fixed_width

In [0]:
def format_excel(full_file_name, dic_col_names, dic_col_fixed_width):
    import math
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation

    wb = openpyxl.load_workbook(full_file_name)
    sheet = wb.active

    dic_col_size = {}

    data_validation_error_title = "Valor Inválido"
    data_validation_error_message = "Este valor não corresponde às restrições de valições de dados definidas para esta célula."
    
    dv_achado = DataValidation(type="list", formula1='"1 - Sim (Tem Doença Colon),2 - Sim (Mas Não Tem Doença Colon),3 - Não"', allow_blank=True)
    dv_achado.showInputMessage = True
    dv_achado.showErrorMessage = True
    dv_achado.error = data_validation_error_message
    dv_achado.errorTitle = data_validation_error_title

    dv_linha_cuidado = DataValidation(type="list", formula1='"1 - doenca 01,2 - doenca 02,3 - doenca 03,4 - doenca 04,5 - doenca 05,6 - Outros"', allow_blank=True)
    dv_linha_cuidado.showInputMessage = True
    dv_linha_cuidado.showErrorMessage = True
    dv_linha_cuidado.error = data_validation_error_message
    dv_linha_cuidado.errorTitle = data_validation_error_title

    dv_navegacao = DataValidation(type="list", formula1='"1 - Navegação Transplante,2 - Navegação Origem Gastro,3 - Navegação Origem Hepato,4 - Navegação Outros,5 - Navegação Oncologia,6 - Não"', allow_blank=True)
    dv_navegacao.showInputMessage = True
    dv_navegacao.showErrorMessage = True
    dv_navegacao.error = data_validation_error_message
    dv_navegacao.errorTitle = data_validation_error_title

    dv_prioridade = DataValidation(type="list", formula1='"1,2,3"', allow_blank=True)
    dv_prioridade.showInputMessage = True
    dv_prioridade.showErrorMessage = True
    dv_prioridade.error = data_validation_error_message
    dv_prioridade.errorTitle = data_validation_error_title

    # col_idade = ""

    wrap_text = [
        "Laudo",
        "Tipo Exame",
        "Nome Hospital",
    ]

    h_align = [
        "Idade Paciente",
        "UF Hospital",
        "Data Exame",
        # "Valor PLT",
        # "Dif. Tempo Imagem PLT",
        "Achado Relevante",
        "Linha de Cuidado",
        # "CRM",
        # "UF CRM",
        "Prioridade"
    ]

    for cols in sheet.iter_cols():
        for cell in cols:
            # Renomeia as colunas
            if cell.row == 1:
                cell.value = dic_col_names.get(cell.value, cell.value)

            column_letter = cell.column_letter
            len_value = len(str(cell.value))
            
            if len_value > dic_col_size.get(column_letter, 0):
                dic_col_size[column_letter] = len_value

            if cols[0].value == "Achado Relevante" and cell.row > 1:
                dv_achado.add(cell)

            if cols[0].value == "Linha de Cuidado" and cell.row > 1:
                dv_linha_cuidado.add(cell)

            if cols[0].value == "Destino" and cell.row > 1:
                dv_navegacao.add(cell)

            if cols[0].value == "Prioridade" and cell.row > 1:
                dv_prioridade.add(cell)

            # if cols[0].value == "Idade Paciente":
            #     if cell.row > 1:
            #         idade = cell.value
            #         if idade > IDADE_LIMITE:
            #             cell.fill = PatternFill(fill_type="solid", start_color="F2DCDB")
            #             cell.font = Font(color="9C0006")

            alignment = copy.copy(cell.alignment)
            alignment.vertical = "center"

            if cell.row > 1 and cols[0].value in wrap_text:
                alignment.wrapText = True

            if cell.row > 1 and cols[0].value in h_align:
                alignment.horizontal = "center"
        
            cell.alignment = alignment

    # Ajusta a largura das colunas
    for k, v in dic_col_size.items():
        adjusted_width = dic_col_fixed_width.get(sheet[f"{k}1"].value)

        if adjusted_width is None:
            adjusted_width = int((v + 2))

        if adjusted_width == -1:
            sheet.column_dimensions[k].hidden = True
        else:
            sheet.column_dimensions[k].width = adjusted_width
                
    sheet.add_data_validation(dv_achado)
    sheet.add_data_validation(dv_linha_cuidado)
    sheet.add_data_validation(dv_navegacao)
    sheet.add_data_validation(dv_prioridade)

    wb.save(full_file_name)
    wb.close()


In [0]:
# def format_excel(full_file_name, dic_col_names, dic_col_fixed_width):
#     import math
#     from openpyxl.styles import Alignment, Font, PatternFill
#     from openpyxl.worksheet.datavalidation import DataValidation

#     wb = openpyxl.load_workbook(full_file_name)
#     sheet = wb.active

#     dic_col_size = {}

#     # col_idade = ""

#     wrap_text = [
#         "Laudo",
#         "Tipo Exame",
#         "Nome Hospital",
#     ]

#     h_align = [
#         "Idade Paciente",
#         "Exame",
#         "Data Exame",
#         "Data Referência",
#         "Origem Exame",
#     ]

#     for cols in sheet.iter_cols():
#         for cell in cols:
#             # Renomeia as colunas
#             if cell.row == 1:
#                 cell.value = dic_col_names.get(cell.value, cell.value)

#             column_letter = cell.column_letter
#             len_value = len(str(cell.value))
            
#             if len_value > dic_col_size.get(column_letter, 0):
#                 dic_col_size[column_letter] = len_value

#             alignment = copy.copy(cell.alignment)
#             alignment.vertical = "center"

#             if cell.row > 1 and cols[0].value in wrap_text:
#                 alignment.wrapText = True

#             if cell.row > 1 and cols[0].value in h_align:
#                 alignment.horizontal = "center"
        
#             cell.alignment = alignment

#     # Ajusta a largura das colunas
#     for k, v in dic_col_size.items():
#         adjusted_width = dic_col_fixed_width.get(sheet[f"{k}1"].value)

#         if adjusted_width is None:
#             adjusted_width = int((v + 2))

#         if adjusted_width == -1:
#             sheet.column_dimensions[k].hidden = True
#         else:
#             sheet.column_dimensions[k].width = adjusted_width

#     wb.save(full_file_name)
#     wb.close()


In [0]:
def copy_files(source, target):
    source = f"file://{source}"
    # print("-"*120)
    print("Copiando arquivo")
    print(f"De  : {source}")
    print(f"Para: {target}")

    dbutils.fs.cp(source, target, recurse=True)

In [0]:
def extract_data(unidade, ids):
    print("-" * 120)
    print(f"Extraindo dados para a unidade: {unidade}")
    
    _df_export = get_data(ids)

    print(f"Registros encontrados: {len(_df_export.index)}")

    _dic_col_names = get_cols()
    
    _info = export_to_excel(_df_export, _dic_col_names, unidade)
    
    _dic_col_fixed_width = get_cols_config()
    
    format_excel(_info['arquivoLocal'], _dic_col_names, _dic_col_fixed_width)
    
    copy_files(_info['arquivoLocal'], _info['arquivoRemoto'])

    return _info

In [0]:
def clean(folder):
    files = os.listdir(folder)
    for file in files:
        file_path = os.path.join(folder, file)        
        if os.path.isfile(file_path):
            print(f"Excluido arquivo: {file_path}")
            os.remove(file_path)

In [0]:
df = spark.read.option("multiline","true").json(f"{root_remote_config_path}unidades.json")
df.display()

In [0]:
files = []

for row in df.collect():
    if row.unidades is None:
        print("-" * 120)
        print(f"Unidade: {row.unidade} - Não configurada!")
        continue

    info = extract_data(row.unidade, row.unidades)
    files.append(info)

In [0]:
metadados = f"{current_folder}{id_projeto}_metadados_envio.json"

json_data = json.dumps(files)

with open(metadados, "w") as file:
    file.write(json_data)

In [0]:
copy_files(metadados, root_remote_data_path)

In [0]:
clean(current_folder)

In [0]:
!ls -lha {current_folder}

In [0]:
!rm -rf {current_folder}

# Envio de arquivos para o OneDrive

#Dataclass Info

In [0]:
@dataclass
class Info:
    fileName: str = ""
    sourcePath: str = ""
    targetPath: str = ""
    environment: str = ""
    unidade: str = ""
    onedriveUser: str = ""
    logicAppUrl: str = ""
    emailToDefault: str = ""
    emailTo: str = ""
    emailCc: str = ""
    emailBcc: str = ""
    emailSubject: str = ""
    emailBody: str = ""
    registros: str = ""
    dataProcessamento: str = ""
    dataProcessamentoFormatada: str = ""

#Funções

In [0]:
def load_metadata(file_path):
    """
    Carrega metadados de um arquivo JSON e cria uma lista de objetos Info.

    Args:
        file_path (str): O caminho do arquivo JSON contendo os metadados.

    Returns:
        list: Uma lista de objetos Info preenchidos com os dados do arquivo JSON.
    """
    info_list = []

    df = spark.read.option("multiline", "true").json(file_path)

    for row in df.collect():
        info_list.append(
            Info(
                environment=environment,
                fileName=row["nomeArquivo"],
                sourcePath=row["caminhoRemoto"],
                unidade=row["unidade"],
                registros=row["registros"],
                dataProcessamento=row["dataProcessamento"],
                dataProcessamentoFormatada=row["dataProcessamentoFormatada"],
            )
        )

    return info_list

In [0]:
def load_config(file_path, info_list, target_path_suffix):
    """
    Carrega a configuração de um arquivo JSON e atualiza a lista de objetos Info.

    Args:
        file_path (str): O caminho do arquivo JSON contendo a configuração.
        info_list (list): Uma lista de objetos Info a serem atualizados.

    Returns:
        list: A lista de objetos Info atualizada com os dados da configuração.
    """
    df = spark.read.option("multiline", "true").json(file_path)
    display(df)

    for row in df.collect():
        for info in info_list:
            info.targetPath = f"{row['targetPath']}{target_path_suffix}"
            info.onedriveUser = row["onedriveUser"]
            info.logicAppUrl = row["logicAppUrl"]
            info.emailToDefault = row["emailToDefault"]

            _subject = "emailSubject" if info.registros > 0 else "emailSubjectNoRecords"
            _body = "emailBody" if info.registros > 0 else "emailBodyNoRecords"

            info.emailSubject = f"{'[' + environment.upper() + ']' if environment != 'prd' else ''}{row[_subject]}"
            info.emailBody = row[_body]

        break

    return info_list

In [0]:
def load_unidades(file_path, info_list):
    """
    Carrega informações de unidades de um arquivo JSON e atualiza a lista de objetos Info.

    Args:
        file_path (str): O caminho do arquivo JSON contendo as informações das unidades.
        info_list (list): Uma lista de objetos Info a serem atualizados.

    Returns:
        list: A lista de objetos Info atualizada com os dados das unidades.
    """
    df = spark.read.option("multiline", "true").json(file_path)

    for info in info_list:
        df_filtered = df.filter(df.unidade == info.unidade)

        # if not df_filtered.rdd.isEmpty():
        if df_filtered.count() > 0:
            row = df_filtered.collect()[0]

            info.emailTo = row["emailTo"]
            info.emailCc = row["emailCc"]
            info.emailBcc = row["emailBcc"]
        else:
            info.emailTo = info.emailToDefault

    return info_list

In [0]:
def send_to_onedrive(info_list):
    """
    Envia informações para o OneDrive usando a URL da Logic App.

    Args:
        info_list (list): Uma lista de objetos Info contendo as informações a serem enviadas.

    Returns:
        None
    """
    for info in info_list:
        print("-" * 120)
        print(f"Enviando: {info.unidade} - Registros: {info.registros}")
        payload = json.dumps(asdict(info))

        response = requests.post(
            info.logicAppUrl, data=payload, headers={"Content-Type": "application/json"}
        )

#Carga das configurações

In [0]:
file_path = f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/data/{environment}/envio/{id_projeto}_metadados_envio.json"
info_list = load_metadata(file_path)
print(info_list)

In [0]:
config_path = f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/config_v2.json"
info_list = load_config(config_path, info_list, "Envio/")
print(info_list)

In [0]:
unidades_path = f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/unidades.json"
info_list = load_unidades(unidades_path, info_list)
print(info_list)

#Envia arquivos

In [0]:
send_to_onedrive(info_list)

In [0]:
dbutils.notebook().exit("Fim da execução!")

In [0]:
%sql
select *
from diamond_hepatologia.hepatologia.vw_diamond_mod_hepatologia
limit 100